# First shift · Endpoint triage

You are covering the service desk. An endpoint scan flags several laptops. Your assignment is to identify the **Windows laptop with disk encryption turned off**, open one incident for its owner, and leave a handoff for the endpoint team.

Work down the page. Run each code cell with its play button or Shift+Enter. Two cells contain choices for you to edit. Use fictional lab details throughout. Everything runs against a temporary database in your notebook session.

## 1 · Start your shift
This loads the published Python simulator and seeds a new practice database. Rerun this cell whenever you want to start over. In Colab, connect to a Python runtime first.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from urllib.request import urlopen
import hashlib
import re
import sqlite3
import subprocess
import sys

revision = "293ef4724d55458cfcd656b1405ab536efc26540"
expected = "8ca7e320e07c36f09a9c9c412e64d4636a423df22742b84b89afd013154bbd37"
source_url = f"https://raw.githubusercontent.com/masterchris101/it-service-desk-toolkit/{revision}/it_helpdesk_toolkit.py"
local = Path("it_helpdesk_toolkit.py")
if local.is_file() and hashlib.sha256(local.read_bytes()).hexdigest() == expected:
    source = local.read_bytes()
else:
    with urlopen(source_url, timeout=30) as response:
        source = response.read()
if hashlib.sha256(source).hexdigest() != expected:
    raise RuntimeError("Simulator download did not match the lab version. Rerun setup.")

session = TemporaryDirectory()
lab = Path(session.name)
script = lab / "it_helpdesk_toolkit.py"
script.write_bytes(source)

def desk(*args):
    result = subprocess.run([sys.executable, str(script), *map(str, args)],
                            cwd=lab, capture_output=True, text=True, check=True)
    return result.stdout.strip()

def records(query, params=()):
    with sqlite3.connect(lab / "helpdesk.db") as db:
        db.row_factory = sqlite3.Row
        return db.execute(query, params).fetchall()

print(desk("seed"))

## 2 · Read the queue
Run the scan. Look for the device whose encryption check failed and find its owner. Several devices fail for different reasons.

In [ ]:
print(desk("devices", "compliance"))

## 3 · Open the incident
Replace the two empty strings with the device and owner from the report. The check gives feedback on your choice. Rerunning a correct answer reuses your incident.

In [ ]:
device = ""  # Example format: WIN-LT-000
owner = ""   # Use the owner listed in parentheses.
device = device.strip().upper()
owner = owner.strip().lower()

def open_incident(device, owner):
    matches = records("SELECT * FROM devices WHERE name = ?", (device,))
    if not matches:
        print("Look up the full device ID in the scan above and try again.")
        return
    endpoint = matches[0]
    if endpoint["os"] != "Windows" or endpoint["encrypted"]:
        print("This device does not match the assignment. Check the reason under its name.")
        return
    if owner != endpoint["owner"]:
        print("Correct device. Check who owns it in the scan.")
        return
    summary = f"{device}: disk encryption disabled"
    prior = records("SELECT id FROM tickets WHERE short_desc=? AND caller=?", (summary, owner))
    if prior:
        number = prior[0]["id"]
    else:
        output = desk("ticket", "new", summary, "--user", owner, "--impact", "3",
                      "--urgency", "2", "--category", "Endpoint Management")
        number = int(re.search(r"INC(\d+)", output).group(1))
    print(f"Incident INC{number:07d} is assigned to your queue.")
    return number

ticket_id = open_incident(device, owner)

## 4 · Hand off the ticket
Enter a short note that names the device, describes the encryption finding, and asks the endpoint team for a concrete next action. The device still needs attention, so move the incident to **In Progress**. Rerun this cell if you revise your note.

In [ ]:
handoff = ""  # Write the note the next technician will read.

if ticket_id is None:
    print("Finish step 3 first.")
elif not handoff.strip():
    print("Write a handoff note before updating the ticket.")
else:
    previous = records("SELECT notes FROM tickets WHERE id=?", (ticket_id,))[0]["notes"]
    last_note = previous.rsplit("] ", 1)[-1]
    if handoff.strip() != last_note:
        print(desk("ticket", "update", ticket_id, "--state", "In Progress", "--note", handoff.strip()))
    else:
        print("Your handoff is already saved.")

## 5 · Check your work
The check reads what the simulator actually saved. Read your note back as if you were the next technician. Would you know what to check and who should confirm the result?

In [ ]:
def check_shift():
    if ticket_id is None:
        print("No incident selected. Return to step 3.")
        return
    row = records("SELECT * FROM tickets WHERE id=?", (ticket_id,))[0]
    checks = {
        "Device and owner recorded": device in row["short_desc"] and row["caller"] == owner,
        "Queued for endpoint management": row["category"] == "Endpoint Management",
        "Incident in progress": row["state"] == "In Progress",
        "Handoff saved": bool(row["notes"].strip()),
    }
    for label, passed in checks.items():
        print(f"{'PASS' if passed else 'REVISIT'}  {label}")
    print("\nSaved technician notes:", row["notes"] or "(none)")
    if all(checks.values()):
        print("\nShift complete. The endpoint team can now pick up this incident.")

check_shift()

## Tell the maintainer how it went
If you tried the lab, [share first-hand feedback on GitHub](https://github.com/masterchris101/it-service-desk-toolkit/issues/new?title=Endpoint+triage+lab+feedback&body=I+tried+the+Endpoint+Triage+lab.%0A%0AEnvironment+%28Colab+or+local%29%3A%0ALast+step+I+reached%3A%0AWhat+worked+or+where+I+got+stuck%3A%0AOne+thing+I+would+change%3A%0A%0APlease+use+fictional+lab+details+only.). A step number and the exact confusing instruction are enough. You can report a blocker even if you stopped at setup. The notebook sends no results automatically.